# LeetCode 1128 — Number of Equivalent Domino Pairs bằng Python

**Link đề:** https://leetcode.com/problems/number-of-equivalent-domino-pairs/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Counting

```python
class Solution:
    def numEquivDominoPairs(self, dominoes: List[List[int]]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **tương đương** (equivalence) và kỹ thuật **chuẩn hóa** (normalize): đưa mọi phần tử tương đương về **một dạng đại diện**.
2. Dùng `tuple` làm **key của dict** (list thì không được — vì sao?).
3. Đếm cặp bằng công thức tổ hợp `k·(k-1)/2` và bằng kỹ thuật **đếm trực tuyến** ("mỗi phần tử mới tạo cặp với tất cả phần tử tương đương đã gặp").
4. Tối ưu thêm bằng **mã hóa thành số** và mảng đếm 100 ô.
5. Trace, chứng minh tính đúng, Time/Space, bộ test đối chiếu, biến thể và bài tập.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — đếm cặp domino tương đương

Cho danh sách `dominoes`, mỗi quân `dominoes[i] = [a, b]`. Hai quân `[a, b]` và `[c, d]` **tương đương** nếu và chỉ nếu `(a == c và b == d)` **hoặc** `(a == d và b == c)` — tức là xoay một quân thì được quân kia.

Trả về số cặp `(i, j)` với `0 <= i < j < len(dominoes)` mà `dominoes[i]` tương đương `dominoes[j]`.

| dominoes | Output | Các cặp |
|---|---:|---|
| `[[1,2],[2,1],[3,4],[5,6]]` | 1 | `(0,1)` |
| `[[1,2],[1,2],[1,1],[1,2],[2,2]]` | 3 | `(0,1)`, `(0,3)`, `(1,3)` |

**Constraints:** `1 <= dominoes.length <= 4·10^4`; `dominoes[i].length == 2`; `1 <= dominoes[i][j] <= 9`.

**Gợi ý của đề:** với mỗi quân `j`, đếm số quân `i < j` đã gặp mà tương đương với nó; dùng hashmap lưu những gì đã gặp.

In [1]:
def equivalent(d1, d2):
    a, b2 = d1
    c, d = d2
    return (a == c and b2 == d) or (a == d and b2 == c)

print(equivalent([1, 2], [2, 1]), equivalent([1, 2], [1, 2]), equivalent([1, 2], [1, 3]), equivalent([1, 1], [1, 1]))

True True False True


Ở cell trên, `a, b2 = d1` là **unpacking**: tách list 2 phần tử vào 2 biến.

## 2. Nền tảng

### 2.1. Chuẩn hóa — chọn một "đại diện" cho mỗi lớp tương đương

`[1,2]` và `[2,1]` là "cùng một quân". Nếu ta luôn viết quân với **số nhỏ trước**: `(min(a,b), max(a,b))`, thì hai quân tương đương **khi và chỉ khi** dạng chuẩn của chúng **bằng nhau**. Kiểm tra tương đương phức tạp trở thành phép so sánh bằng đơn giản.

In [2]:
def normalize(d):
    a, b2 = d
    return (min(a, b2), max(a, b2))

for d in [[1, 2], [2, 1], [3, 3], [9, 4]]:
    print(d, "->", normalize(d))
print(normalize([1, 2]) == normalize([2, 1]))

[1, 2] -> (1, 2)
[2, 1] -> (1, 2)
[3, 3] -> (3, 3)
[9, 4] -> (4, 9)
True


### 2.2. Tuple làm key của dict — vì sao không dùng list?

Key của `dict`/phần tử của `set` phải **hashable** (băm được). Đối tượng **bất biến** như `int`, `str`, `tuple` (chứa phần tử bất biến) là hashable. `list` có thể thay đổi nên **không** hashable — nếu nội dung đổi sau khi đã làm key, giá trị băm sẽ sai.

In [3]:
counts = {}
counts[(1, 2)] = 3
print(counts, "| hash((1,2)) chạy được:", isinstance(hash((1, 2)), int))
try:
    counts[[1, 2]] = 3
except TypeError as e:
    print("Lỗi khi dùng list làm key:", e)
print("tuple([2, 1]) =", tuple([2, 1]), "| tuple(sorted([2, 1])) =", tuple(sorted([2, 1])))

{(1, 2): 3} | hash((1,2)) chạy được: True
Lỗi khi dùng list làm key: unhashable type: 'list'
tuple([2, 1]) = (2, 1) | tuple(sorted([2, 1])) = (1, 2)


### 2.3. Đếm số cặp trong một nhóm: `k·(k-1)/2`

Nếu một nhóm có `k` quân giống nhau, số cách chọn 2 quân (không thứ tự) là tổ hợp `C(k, 2) = k·(k-1)/2`.

| k | cặp |
|---:|---:|
| 1 | 0 |
| 2 | 1 |
| 3 | 3 |
| 4 | 6 |

Một cách nhìn khác (sẽ dùng ở cách đếm trực tuyến): quân thứ 2 ghép với 1 quân trước, quân thứ 3 ghép với 2 quân trước, … → `0 + 1 + 2 + ... + (k-1) = k(k-1)/2`.

In [4]:
from itertools import combinations
for k in range(1, 6):
    items = list(range(k))
    print(f"k={k}: số cặp liệt kê = {len(list(combinations(items, 2)))}, công thức = {k * (k - 1) // 2}, cộng dồn = {sum(range(k))}")

k=1: số cặp liệt kê = 0, công thức = 0, cộng dồn = 0
k=2: số cặp liệt kê = 1, công thức = 1, cộng dồn = 1
k=3: số cặp liệt kê = 3, công thức = 3, cộng dồn = 3
k=4: số cặp liệt kê = 6, công thức = 6, cộng dồn = 6
k=5: số cặp liệt kê = 10, công thức = 10, cộng dồn = 10


`itertools.combinations(items, 2)` sinh mọi cặp `(items[i], items[j])` với `i < j` — tiện để kiểm chứng công thức.

## 3. Cách 1 — Brute Force: thử mọi cặp `i < j`

In [5]:
from typing import List

def num_pairs_brute(dominoes: List[List[int]]) -> int:
    n = len(dominoes)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if equivalent(dominoes[i], dominoes[j]):
                count += 1
    return count

print(num_pairs_brute([[1, 2], [2, 1], [3, 4], [5, 6]]))               # 1
print(num_pairs_brute([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]))       # 3

1
3


### Nút thắt

`n(n-1)/2` cặp. Với `n = 4·10^4` là khoảng `8·10^8` phép so sánh → quá chậm (Time Limit Exceeded).

Quan sát quan trọng: giá trị chỉ từ 1 đến 9 → chỉ có **45** loại quân khác nhau sau chuẩn hóa (`9` quân đôi + `C(9,2) = 36` quân khác số). Có `40 000` quân nhưng chỉ 45 "loại" → rất nhiều quân trùng loại. Thay vì so từng cặp, hãy **đếm số quân mỗi loại**.

In [6]:
kinds = {(a, b2) for a in range(1, 10) for b2 in range(a, 10)}
print("Số loại quân sau chuẩn hóa:", len(kinds))
n = 4 * 10**4
print(f"Brute force với n={n}: {n * (n - 1) // 2:,} cặp")

Số loại quân sau chuẩn hóa: 45
Brute force với n=40000: 799,980,000 cặp


## 4. Insight — chuẩn hóa rồi đếm theo nhóm

1. Chuẩn hóa từng quân về `(min, max)`.
2. Đếm số lần xuất hiện của mỗi dạng chuẩn: `freq[key]`.
3. Mỗi nhóm `k` quân cho `k(k-1)/2` cặp; tổng tất cả nhóm là đáp án.

Hai quân thuộc **khác nhóm** thì không tương đương, nên không bị đếm; hai quân **cùng nhóm** luôn tương đương, nên đếm đủ.

In [7]:
dominoes = [[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]
freq = {}
for d in dominoes:
    key = normalize(d)
    freq[key] = freq.get(key, 0) + 1
print("freq =", freq)
print("cặp mỗi nhóm:", {k: v * (v - 1) // 2 for k, v in freq.items()})
print("tổng =", sum(v * (v - 1) // 2 for v in freq.values()))

freq = {(1, 2): 3, (1, 1): 1, (2, 2): 1}
cặp mỗi nhóm: {(1, 2): 3, (1, 1): 0, (2, 2): 0}
tổng = 3


`dict.get(key, 0)` trả giá trị của `key` nếu có, ngược lại trả `0` — tránh lỗi `KeyError` khi gặp key lần đầu. `dict.items()` cho các cặp `(key, value)`, `dict.values()` cho các value.

## 5. Cách 2 — Đếm nhóm + công thức `k(k-1)/2` (dùng `Counter`)

`collections.Counter(iterable)` đếm hộ ta: mỗi phần tử khác nhau thành một key.

In [8]:
from collections import Counter

def num_pairs_counter(dominoes: List[List[int]]) -> int:
    freq = Counter((min(a, b2), max(a, b2)) for a, b2 in dominoes)
    return sum(k * (k - 1) // 2 for k in freq.values())

print(num_pairs_counter([[1, 2], [2, 1], [3, 4], [5, 6]]))
print(num_pairs_counter([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]))

1
3


## 6. Cách 3 — Đếm trực tuyến một lượt (lời giải chính, đúng gợi ý)

Duyệt từ trái sang phải. Khi gặp quân `j` có dạng chuẩn `key`:
- Nó tạo cặp với **mọi** quân `i < j` cùng `key` đã gặp — có đúng `seen[key]` quân như vậy → `ans += seen[key]`.
- Sau đó mới ghi nhận nó: `seen[key] += 1`.

Thứ tự "**cộng trước, tăng sau**" bảo đảm không ghép quân với chính nó (giống Two Sum: kiểm tra trước, lưu sau).

In [9]:
def num_pairs_online(dominoes: List[List[int]]) -> int:
    seen = {}
    ans = 0
    for a, b2 in dominoes:
        key = (a, b2) if a <= b2 else (b2, a)
        ans += seen.get(key, 0)        # ghép với mọi quân tương đương đã gặp
        seen[key] = seen.get(key, 0) + 1
    return ans

print(num_pairs_online([[1, 2], [2, 1], [3, 4], [5, 6]]))
print(num_pairs_online([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]))

1
3


## 7. TRACE từng bước

Ví dụ 2: `[[1,2],[1,2],[1,1],[1,2],[2,2]]`.

| j | quân | key | `seen[key]` trước | ans += | ans | seen sau |
|---:|---|---|---:|---:|---:|---|
| 0 | [1,2] | (1,2) | 0 | 0 | 0 | {(1,2):1} |
| 1 | [1,2] | (1,2) | 1 | 1 | 1 | {(1,2):2} |
| 2 | [1,1] | (1,1) | 0 | 0 | 1 | {(1,2):2,(1,1):1} |
| 3 | [1,2] | (1,2) | 2 | 2 | 3 | {(1,2):3,(1,1):1} |
| 4 | [2,2] | (2,2) | 0 | 0 | 3 | {…,(2,2):1} |

Ở `j = 3`, quân `[1,2]` ghép với 2 quân `[1,2]` trước nó (vị trí 0 và 1) → cặp `(0,3)` và `(1,3)`.

In [10]:
def trace_num_pairs(dominoes):
    seen = {}
    ans = 0
    for j, (a, b2) in enumerate(dominoes):
        key = (a, b2) if a <= b2 else (b2, a)
        before = seen.get(key, 0)
        ans += before
        seen[key] = before + 1
        print(f"j={j}, quân={[a, b2]}, key={key}, đã gặp {before} quân cùng loại -> ans={ans} | seen={seen}")
    print("Kết quả:", ans)
    return ans

_ = trace_num_pairs([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]])

j=0, quân=[1, 2], key=(1, 2), đã gặp 0 quân cùng loại -> ans=0 | seen={(1, 2): 1}
j=1, quân=[1, 2], key=(1, 2), đã gặp 1 quân cùng loại -> ans=1 | seen={(1, 2): 2}
j=2, quân=[1, 1], key=(1, 1), đã gặp 0 quân cùng loại -> ans=1 | seen={(1, 2): 2, (1, 1): 1}
j=3, quân=[1, 2], key=(1, 2), đã gặp 2 quân cùng loại -> ans=3 | seen={(1, 2): 3, (1, 1): 1}
j=4, quân=[2, 2], key=(2, 2), đã gặp 0 quân cùng loại -> ans=3 | seen={(1, 2): 3, (1, 1): 1, (2, 2): 1}
Kết quả: 3


`for j, (a, b2) in enumerate(...)`: `enumerate` cho `(j, quân)`, và `(a, b2)` tách tiếp quân thành 2 số — unpacking lồng nhau.

Trace ví dụ 1 — để thấy quân xoay `[2,1]` được chuẩn hóa thành `(1,2)`:

In [11]:
_ = trace_num_pairs([[1, 2], [2, 1], [3, 4], [5, 6]])

j=0, quân=[1, 2], key=(1, 2), đã gặp 0 quân cùng loại -> ans=0 | seen={(1, 2): 1}
j=1, quân=[2, 1], key=(1, 2), đã gặp 1 quân cùng loại -> ans=1 | seen={(1, 2): 2}
j=2, quân=[3, 4], key=(3, 4), đã gặp 0 quân cùng loại -> ans=1 | seen={(1, 2): 2, (3, 4): 1}
j=3, quân=[5, 6], key=(5, 6), đã gặp 0 quân cùng loại -> ans=1 | seen={(1, 2): 2, (3, 4): 1, (5, 6): 1}
Kết quả: 1


## 8. Cách 4 — Mã hóa thành số + mảng đếm 100 ô

Vì `1 <= a, b <= 9`, dạng chuẩn `(lo, hi)` có thể mã hóa thành **một số** `lo * 10 + hi` (từ 11 đến 99). Dùng list `count = [0] * 100` thay cho dict: truy cập chỉ số nhanh hơn băm tuple.

In [12]:
for d in [[1, 2], [2, 1], [9, 4], [3, 3]]:
    lo, hi = min(d), max(d)
    print(d, "-> mã", lo * 10 + hi)

def num_pairs_array(dominoes: List[List[int]]) -> int:
    count = [0] * 100
    ans = 0
    for a, b2 in dominoes:
        code = a * 10 + b2 if a <= b2 else b2 * 10 + a
        ans += count[code]
        count[code] += 1
    return ans

print(num_pairs_array([[1, 2], [2, 1], [3, 4], [5, 6]]), num_pairs_array([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]))

[1, 2] -> mã 12
[2, 1] -> mã 12
[9, 4] -> mã 49
[3, 3] -> mã 33
1 3


## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def numEquivDominoPairs(self, dominoes: List[List[int]]) -> int:
        count = [0] * 100
        ans = 0
        for a, b in dominoes:
            code = a * 10 + b if a <= b else b * 10 + a
            ans += count[code]
            count[code] += 1
        return ans

sol = Solution()
print(sol.numEquivDominoPairs([[1, 2], [2, 1], [3, 4], [5, 6]]))
print(sol.numEquivDominoPairs([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]))

1
3


Bản dict/Counter (tổng quát, không phụ thuộc giá trị `<= 9`):

In [14]:
class SolutionCounter:
    def numEquivDominoPairs(self, dominoes: List[List[int]]) -> int:
        freq = Counter(tuple(sorted(d)) for d in dominoes)
        return sum(k * (k - 1) // 2 for k in freq.values())

print(SolutionCounter().numEquivDominoPairs([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]))

3


## 10. Vì sao đúng? — Độ phức tạp

**Chuẩn hóa đúng:** `[a,b] ~ [c,d]` ⇔ `{a,b}` và `{c,d}` là cùng một **đa tập** 2 phần tử ⇔ `(min(a,b), max(a,b)) == (min(c,d), max(c,d))`. Mã `lo*10 + hi` là đơn ánh vì `1 <= lo, hi <= 9` (một chữ số mỗi bên).

**Đếm trực tuyến đúng (invariant):** trước khi xử lý quân `j`, `seen[key]` = số quân `i < j` có dạng chuẩn `key`, và `ans` = số cặp tương đương `(i, i')` với `i < i' < j`. Khi xử lý `j`, các cặp mới có dạng `(i, j)` với `i < j` tương đương — đúng `seen[key_j]` cặp. Cộng vào rồi tăng `seen[key_j]`, invariant được giữ. Mỗi cặp `(i, j)` được đếm đúng một lần — tại thời điểm xử lý `j`.

**Công thức nhóm đúng:** quan hệ tương đương chia các quân thành các nhóm rời nhau; cặp tương đương = cặp cùng nhóm; nhóm `k` phần tử có `C(k,2)` cặp.

Gọi `n = len(dominoes)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute Force | `O(n²)` | `O(1)` |
| 2. Counter + `k(k-1)/2` | `O(n)` trung bình | `O(45)` = `O(1)` (tối đa 45 key) |
| 3. Đếm trực tuyến (dict) | `O(n)` trung bình | `O(1)` |
| 4. Mảng 100 ô | `O(n)` | `O(1)` |

## 11. Bộ test — mẫu, biên, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS {actual}")

cases = [
    ("mẫu 1", [[1, 2], [2, 1], [3, 4], [5, 6]], 1),
    ("mẫu 2", [[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]], 3),
    ("1 quân", [[4, 7]], 0),
    ("2 quân xoay", [[9, 1], [1, 9]], 1),
    ("quân đôi giống nhau", [[5, 5], [5, 5], [5, 5]], 3),
    ("không cặp nào", [[1, 2], [1, 3], [2, 3]], 0),
    ("4 quân cùng loại", [[2, 3], [3, 2], [2, 3], [3, 2]], 6),
    ("2 nhóm", [[1, 2], [2, 1], [3, 4], [4, 3], [3, 4]], 1 + 3),
]
for name, d, exp in cases:
    check(name, Solution().numEquivDominoPairs(d), exp)

mẫu 1                          PASS 1
mẫu 2                          PASS 3
1 quân                         PASS 0
2 quân xoay                    PASS 1
quân đôi giống nhau            PASS 3
không cặp nào                  PASS 0
4 quân cùng loại               PASS 6
2 nhóm                         PASS 4


In [16]:
import random
rng = random.Random(1128)
funcs = [num_pairs_brute, num_pairs_counter, num_pairs_online, num_pairs_array,
         Solution().numEquivDominoPairs, SolutionCounter().numEquivDominoPairs]
for _ in range(1000):
    d = [[rng.randint(1, 4), rng.randint(1, 4)] for _ in range(rng.randint(1, 30))]
    expected = num_pairs_brute(d)
    for f in funcs:
        assert f(d) == expected, (f, d)
print("1000 bộ ngẫu nhiên: 6 cách khớp Brute Force")

big = [[rng.randint(1, 9), rng.randint(1, 9)] for _ in range(4 * 10**4)]
print("n = 40 000:", Solution().numEquivDominoPairs(big), "==", num_pairs_counter(big))

1000 bộ ngẫu nhiên: 6 cách khớp Brute Force
n = 40 000: 18689621 == 18689621


## 12. Lỗi người mới hay gặp

### Lỗi 1: Không chuẩn hóa — dùng thẳng `(a, b)` làm key

In [17]:
def wrong_no_normalize(dominoes):
    seen, ans = {}, 0
    for a, b2 in dominoes:
        key = (a, b2)                    # SAI: (1,2) và (2,1) là 2 key khác nhau
        ans += seen.get(key, 0)
        seen[key] = seen.get(key, 0) + 1
    return ans

d = [[1, 2], [2, 1], [3, 4], [5, 6]]
print("sai:", wrong_no_normalize(d), "| đúng:", num_pairs_online(d))

sai: 0 | đúng: 1


### Lỗi 2: Tăng `seen` trước rồi mới cộng

In [18]:
def wrong_order(dominoes):
    seen, ans = {}, 0
    for a, b2 in dominoes:
        key = (min(a, b2), max(a, b2))
        seen[key] = seen.get(key, 0) + 1   # SAI: tăng trước
        ans += seen[key]                   # -> tính cả cặp (j, j)
    return ans

d = [[1, 2], [2, 1], [3, 4], [5, 6]]
print("sai:", wrong_order(d), "| đúng:", num_pairs_online(d))

sai: 5 | đúng: 1


**Vì sao sai:** mỗi quân tự ghép với chính nó, nên kết quả dư đúng `n`. **Sửa:** cộng `seen[key]` trước, tăng sau (hoặc dùng `seen[key] - 1` sau khi tăng).

### Lỗi 3: Mã hóa `a + b` hoặc `a * b` — không đơn ánh

In [19]:
def wrong_sum_code(dominoes):
    count, ans = {}, 0
    for a, b2 in dominoes:
        code = a + b2                    # SAI: [1,4] và [2,3] cùng tổng 5
        ans += count.get(code, 0)
        count[code] = count.get(code, 0) + 1
    return ans

d = [[1, 4], [2, 3]]
print("sai:", wrong_sum_code(d), "| đúng:", num_pairs_online(d))

sai: 1 | đúng: 0


**Vì sao sai:** mã hóa phải **đơn ánh** (hai loại khác nhau → hai mã khác nhau). `a + b` và `a * b` đều đối xứng nhưng va chạm. `lo * 10 + hi` an toàn vì mỗi số chỉ có một chữ số.

### Lỗi 4: Dùng list làm key

In [20]:
try:
    seen = {}
    seen[sorted([2, 1])] = 1             # sorted trả về LIST
except TypeError as e:
    print("Lỗi:", e, "-> sửa: tuple(sorted(...))")

Lỗi: unhashable type: 'list' -> sửa: tuple(sorted(...))


## 13. Mẹo nhận dạng pattern

- "Đếm số cặp `(i, j)` thỏa quan hệ **bằng nhau sau một phép biến đổi**" → chuẩn hóa + hash map đếm; tránh `O(n²)`.
- Đếm cặp: hoặc `Σ k(k-1)/2` theo nhóm, hoặc đếm trực tuyến `ans += seen[key]; seen[key] += 1`.
- Hai dữ liệu "tương đương khi xoay / đảo / hoán vị" → dạng chuẩn: `sorted`, `min/max`, chuỗi xoay nhỏ nhất…
- Miền giá trị rất nhỏ → mã hóa thành số + mảng đếm.
- Bài cùng họ: #1512 Number of Good Pairs, #49 Group Anagrams, #2001 Number of Pairs of Interchangeable Rectangles, #1010 Pairs of Songs With Total Durations Divisible by 60, #2364 Count Number of Bad Pairs.

## 14. Biến thể

### 14.1. #1512 Number of Good Pairs — trường hợp không cần chuẩn hóa

In [21]:
def num_identical_pairs(nums: List[int]) -> int:
    seen, ans = {}, 0
    for x in nums:
        ans += seen.get(x, 0)
        seen[x] = seen.get(x, 0) + 1
    return ans

print(num_identical_pairs([1, 2, 3, 1, 1, 3]))   # 4

4


### 14.2. #49 Group Anagrams — chuẩn hóa chuỗi bằng `sorted`

Hai từ là anagram ⇔ cùng tập ký tự với cùng số lần ⇔ `"".join(sorted(w))` bằng nhau. Gom nhóm thay vì đếm.

In [22]:
def group_anagrams(words: List[str]) -> List[List[str]]:
    groups = {}
    for w in words:
        key = "".join(sorted(w))
        groups.setdefault(key, []).append(w)
    return list(groups.values())

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


`dict.setdefault(key, default)` trả giá trị của `key`; nếu chưa có thì gán `default` rồi trả nó — tiện để gom vào list.

### 14.3. Liệt kê các cặp (không chỉ đếm)

Lưu **danh sách chỉ số** đã gặp cho mỗi key; mỗi quân mới ghép với tất cả chỉ số trong danh sách. (Số cặp có thể lên tới `O(n²)` nên chỉ dùng khi cần liệt kê.)

In [23]:
def list_equiv_pairs(dominoes):
    seen = {}
    pairs = []
    for j, (a, b2) in enumerate(dominoes):
        key = (min(a, b2), max(a, b2))
        for i in seen.get(key, []):
            pairs.append((i, j))
        seen.setdefault(key, []).append(j)
    return pairs

print(list_equiv_pairs([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]))

[(0, 1), (0, 3), (1, 3)]


## 15. Bài tập (tăng dần)

**Bài 1 — Tính tay:** `[[3,3],[1,3],[3,1],[3,3],[1,3],[3,3]]` có bao nhiêu cặp? Tính bằng cả 2 cách: theo nhóm và theo trace trực tuyến.

**Bài 2 — Dự đoán output:** `Counter(tuple(sorted(d)) for d in [[2,1],[1,2],[2,2]])`.

**Bài 3 — TODO:** viết lời giải đếm trực tuyến bằng dict.

In [24]:
def exercise_num_equiv(dominoes: List[List[int]]) -> int:
    # TODO 1: seen = {}, ans = 0
    # TODO 2: chuẩn hóa key; ans += seen.get(key, 0); seen[key] += 1
    pass

# assert exercise_num_equiv([[1, 2], [2, 1], [3, 4], [5, 6]]) == 1
# assert exercise_num_equiv([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]) == 3

**Bài 4 — Nhóm lớn nhất:** trả về số quân của **nhóm tương đương đông nhất**.

In [25]:
def exercise_largest_group(dominoes: List[List[int]]) -> int:
    # TODO
    pass

# assert exercise_largest_group([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]) == 3

**Bài 5 — Domino 3 mặt:** mỗi quân là `[a, b, c]`, tương đương nếu là **hoán vị** của nhau. Đếm số cặp tương đương.

In [26]:
def exercise_triple_pairs(pieces: List[List[int]]) -> int:
    # TODO: key = tuple(sorted(piece))
    pass

# assert exercise_triple_pairs([[1, 2, 3], [3, 2, 1], [2, 1, 3], [1, 1, 2]]) == 3

**Bài 6 — Mini challenge (#2001):** cho các hình chữ nhật `[w, h]`; hai hình "hoán đổi được" nếu `w1/h1 == w2/h2`. Đếm số cặp. Gợi ý: chuẩn hóa tỉ số bằng **phân số tối giản** `(w // g, h // g)` với `g = gcd(w, h)` để tránh sai số số thực.

In [27]:
def exercise_interchangeable_rectangles(rects: List[List[int]]) -> int:
    # TODO: from math import gcd
    pass

# assert exercise_interchangeable_rectangles([[4, 8], [3, 6], [10, 20], [15, 30]]) == 6
# assert exercise_interchangeable_rectangles([[4, 5], [7, 8]]) == 0

### 15.1. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chuẩn hóa: (3,3)×3, (1,3)×3 | mỗi nhóm 3 quân | `3 + 3 = 6` |
| 2 | `sorted([2,1]) = [1,2]` | `tuple` → `(1,2)` | `Counter({(1, 2): 2, (2, 2): 1})` |
| 3 | mục 6 | cộng trước, tăng sau | `key = (a, b) if a <= b else (b, a)` |
| 4 | Đếm theo nhóm | `Counter` | `max(freq.values())` |
| 5 | Hoán vị ⇔ sorted bằng nhau | key là tuple 3 phần tử | `Σ k(k-1)/2` |
| 6 | `w/h` là số thực → có thể sai số | `g = gcd(w, h)` | key `(w // g, h // g)` |

## 16. Đáp án tham khảo

- **Bài 1:** nhóm `(3,3)` có 3 quân → 3 cặp; nhóm `(1,3)` có 3 quân → 3 cặp; tổng **6**.
- **Bài 2:** `Counter({(1, 2): 2, (2, 2): 1})`.

In [28]:
from math import gcd

def answer_num_equiv(dominoes):
    seen, ans = {}, 0
    for a, b2 in dominoes:
        key = (a, b2) if a <= b2 else (b2, a)
        ans += seen.get(key, 0)
        seen[key] = seen.get(key, 0) + 1
    return ans

def answer_largest_group(dominoes):
    freq = Counter((min(a, b2), max(a, b2)) for a, b2 in dominoes)
    return max(freq.values())

def answer_triple_pairs(pieces):
    freq = Counter(tuple(sorted(p)) for p in pieces)
    return sum(k * (k - 1) // 2 for k in freq.values())

def answer_interchangeable_rectangles(rects):
    seen, ans = {}, 0
    for w, h in rects:
        g = gcd(w, h)
        key = (w // g, h // g)
        ans += seen.get(key, 0)
        seen[key] = seen.get(key, 0) + 1
    return ans

assert answer_num_equiv([[3, 3], [1, 3], [3, 1], [3, 3], [1, 3], [3, 3]]) == 6
assert Counter(tuple(sorted(d)) for d in [[2, 1], [1, 2], [2, 2]]) == Counter({(1, 2): 2, (2, 2): 1})
assert answer_num_equiv([[1, 2], [2, 1], [3, 4], [5, 6]]) == 1
assert answer_largest_group([[1, 2], [1, 2], [1, 1], [1, 2], [2, 2]]) == 3
assert answer_triple_pairs([[1, 2, 3], [3, 2, 1], [2, 1, 3], [1, 1, 2]]) == 3
assert answer_interchangeable_rectangles([[4, 8], [3, 6], [10, 20], [15, 30]]) == 6
assert answer_interchangeable_rectangles([[4, 5], [7, 8]]) == 0
from fractions import Fraction
for _ in range(500):
    rects = [[rng.randint(1, 12), rng.randint(1, 12)] for _ in range(rng.randint(1, 15))]
    brute = sum(1 for (w1, h1), (w2, h2) in combinations(rects, 2) if Fraction(w1, h1) == Fraction(w2, h2))
    assert answer_interchangeable_rectangles(rects) == brute
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


`fractions.Fraction(w, h)` biểu diễn phân số chính xác (không sai số) — ta dùng nó làm "oracle" để kiểm tra đáp án bài 6.

## 17. Template nhớ nhanh

```python
seen, ans = {}, 0
for item in items:
    key = normalize(item)          # đại diện của lớp tương đương
    ans += seen.get(key, 0)        # ghép với mọi phần tử cùng lớp đã gặp
    seen[key] = seen.get(key, 0) + 1
return ans
```

## 18. Checklist submit

- [ ] Đã chuẩn hóa `[a,b]` và `[b,a]` về cùng một key?
- [ ] Key là `tuple`/số (hashable), không phải list?
- [ ] Cộng `seen[key]` **trước**, tăng **sau**?
- [ ] Mã hóa số (nếu dùng) là đơn ánh (`lo*10 + hi`, không phải `a+b`)?
- [ ] Kết quả với 1 quân là 0? Với `k` quân giống nhau là `k(k-1)/2`?